# Held-out Mesh Correction Visual Review

Compare each test phantom's **classical baseline**, **rule-targeted correction**, and **GNN correction** for the clean, moderate, and poor tiers. The notebook reads the completed test report and meshes, creates one three-panel PNG per `(phantom, tier)`, and writes a CSV index under `data/evaluation_reports/figures/mesh_snapshots/`.

The rule-targeted method is the non-learned classical correction comparator. The baseline is the original TetGen mesh. Lower poor-element percentage and FE iterations are better; higher minimum scaled Jacobian is better. A GNN improvement is counted only against `rule_targeted` and is reported as measured, including regressions.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
import pyvista as pv
from IPython.display import display

PROJECT_ROOT = next(
    (path for path in (Path.cwd(), *Path.cwd().parents)
     if (path / "data/evaluation_reports/benchmark_long.csv").is_file()),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Could not locate data/evaluation_reports/benchmark_long.csv from the notebook working directory")

REPORT_PATH = PROJECT_ROOT / "data/evaluation_reports/benchmark_long.csv"
BASELINE_DIR = PROJECT_ROOT / "data/meshes"
CORRECTED_DIR = PROJECT_ROOT / "data/meshes_corrected"
SNAPSHOT_ROOT = PROJECT_ROOT / "data/evaluation_reports/figures/mesh_snapshots"
TIERS = ("clean", "moderate", "poor")
METHODS = ("classical", "rule_targeted", "gnn")
METHOD_LABELS = {
    "classical": "Classical baseline",
    "rule_targeted": "Rule-targeted correction",
    "gnn": "GNN correction",
}
METHOD_COLORS = {
    "classical": "#4C9F9A",
    "rule_targeted": "#E69F00",
    "gnn": "#4C78A8",
}
REQUIRED_METRICS = ("poor_pct", "min_sj", "fe_iters", "status")

results = pd.read_csv(REPORT_PATH, dtype={"id": str})
results["id"] = results["id"].str.zfill(4)
selected = results[results["method"].isin(METHODS)].copy()
missing_columns = sorted(set(REQUIRED_METRICS) - set(selected.columns))
if missing_columns:
    raise ValueError(f"Evaluation report is missing required columns: {missing_columns}")
if selected.empty:
    raise ValueError(f"No evaluation rows found for methods: {METHODS}")

expected = pd.MultiIndex.from_product(
    [sorted(selected["id"].unique()), TIERS, METHODS],
    names=["id", "tier", "method"],
)
observed = pd.MultiIndex.from_frame(selected[["id", "tier", "method"]])
missing_rows = expected.difference(observed)
if len(missing_rows):
    raise ValueError(f"Missing evaluation rows, e.g. {list(missing_rows[:5])}")
failed = selected[selected["status"] != "ok"]
if not failed.empty:
    raise RuntimeError(
        "Cannot render failed evaluation rows: "
        + failed[["id", "tier", "method", "error"]].head().to_string(index=False)
    )

print(f"Project: {PROJECT_ROOT}")
print(f"Test phantoms: {selected['id'].nunique()} | tiers: {', '.join(TIERS)}")
print(f"Panels to render: {len(selected)} | output: {SNAPSHOT_ROOT}")

Project: d:\Mesh Optimization
Test phantoms: 23 | tiers: clean, moderate, poor
Panels to render: 207 | output: d:\Mesh Optimization\data\evaluation_reports\figures\mesh_snapshots


## Render and save snapshots

Each PNG has the same phantom and tier in all three panels. The metric labels come from the corresponding final evaluation rows. Files are grouped by tier and named with the phantom ID.

In [2]:
SNAPSHOT_ROOT.mkdir(parents=True, exist_ok=True)
index_rows = []
phantom_ids = sorted(selected["id"].unique())
total_figures = len(phantom_ids) * len(TIERS)
figure_number = 0

for phantom_id in phantom_ids:
    for tier in TIERS:
        figure_number += 1
        panel_rows = selected[
            (selected["id"] == phantom_id) & (selected["tier"] == tier)
        ].set_index("method")
        mesh_paths = {
            "classical": BASELINE_DIR / f"{phantom_id}_{tier}.vtu",
            "rule_targeted": CORRECTED_DIR / f"{phantom_id}_{tier}_rule_targeted.vtu",
            "gnn": CORRECTED_DIR / f"{phantom_id}_{tier}_gnn.vtu",
        }
        missing_files = [str(path) for path in mesh_paths.values() if not path.is_file()]
        if missing_files:
            raise FileNotFoundError("Snapshot source mesh missing: " + ", ".join(missing_files))

        output_dir = SNAPSHOT_ROOT / tier
        output_dir.mkdir(parents=True, exist_ok=True)
        figure_path = output_dir / f"{phantom_id}.png"
        plotter = pv.Plotter(shape=(1, 3), off_screen=True, window_size=(1800, 680))
        plotter.set_background("white")
        shared_camera = None

        for column, method in enumerate(METHODS):
            plotter.subplot(0, column)
            source_path = mesh_paths[method]
            grid = pv.read(source_path)
            surface = grid.extract_surface().triangulate()
            if surface.n_points == 0 or surface.n_cells == 0:
                plotter.close()
                raise ValueError(f"Empty rendered surface in {source_path}")

            row = panel_rows.loc[method]
            title = (
                f"{METHOD_LABELS[method]}\n"
                f"Poor: {row['poor_pct']:.2f}% | min SJ: {row['min_sj']:.4f} | "
                f"FE iters: {int(row['fe_iters'])}"
            )
            plotter.add_mesh(
                surface,
                color=METHOD_COLORS[method],
                smooth_shading=True,
                show_edges=False,
                lighting=True,
            )
            plotter.add_text(title, position="upper_left", font_size=11, color="black")
            plotter.add_text(
                f"{phantom_id} / {tier}",
                position="lower_left",
                font_size=10,
                color="black",
            )
            if column == 0:
                plotter.view_isometric()
                plotter.reset_camera()
                shared_camera = plotter.camera_position
            else:
                plotter.camera_position = shared_camera

            index_rows.append({
                "figure_path": str(figure_path.relative_to(PROJECT_ROOT)),
                "id": phantom_id,
                "tier": tier,
                "method": method,
                "mesh_path": str(source_path.relative_to(PROJECT_ROOT)),
                "poor_pct": float(row["poor_pct"]),
                "min_sj": float(row["min_sj"]),
                "fe_iters": int(row["fe_iters"]),
            })
        plotter.screenshot(str(figure_path))
        plotter.close()
        print(f"[{figure_number}/{total_figures}] {figure_path.relative_to(PROJECT_ROOT)}")

index_path = SNAPSHOT_ROOT / "snapshot_index.csv"
pd.DataFrame(index_rows).to_csv(index_path, index=False)
if len(index_rows) != len(phantom_ids) * len(TIERS) * len(METHODS):
    raise RuntimeError(f"Expected 3 panels per test phantom and tier; indexed {len(index_rows)} panels")
print(f"Saved {total_figures} comparison images and {len(index_rows)} panel index rows.")
print(f"Index: {index_path.relative_to(PROJECT_ROOT)}")

C:\Users\anshk\AppData\Local\Temp\ipykernel_20792\924178794.py:33: PyVistaFutureWarning: The default value of `algorithm` for the filter
`UnstructuredGrid.extract_surface` will change in the future. It currently defaults to
`'dataset_surface'`, but will change to `None`. Explicitly set the `algorithm` keyword to
silence this warning.
  surface = grid.extract_surface().triangulate()


[1/69] data\evaluation_reports\figures\mesh_snapshots\clean\0006.png


C:\Users\anshk\AppData\Local\Temp\ipykernel_20792\924178794.py:33: PyVistaFutureWarning: The default value of `algorithm` for the filter
`UnstructuredGrid.extract_surface` will change in the future. It currently defaults to
`'dataset_surface'`, but will change to `None`. Explicitly set the `algorithm` keyword to
silence this warning.
  surface = grid.extract_surface().triangulate()


[2/69] data\evaluation_reports\figures\mesh_snapshots\moderate\0006.png


C:\Users\anshk\AppData\Local\Temp\ipykernel_20792\924178794.py:33: PyVistaFutureWarning: The default value of `algorithm` for the filter
`UnstructuredGrid.extract_surface` will change in the future. It currently defaults to
`'dataset_surface'`, but will change to `None`. Explicitly set the `algorithm` keyword to
silence this warning.
  surface = grid.extract_surface().triangulate()


[3/69] data\evaluation_reports\figures\mesh_snapshots\poor\0006.png


... (128 outputs left out)

C:\Users\anshk\AppData\Local\Temp\ipykernel_20792\924178794.py:33: PyVistaFutureWarning: The default value of `algorithm` for the filter
`UnstructuredGrid.extract_surface` will change in the future. It currently defaults to
`'dataset_surface'`, but will change to `None`. Explicitly set the `algorithm` keyword to
silence this warning.
  surface = grid.extract_surface().triangulate()


[68/69] data\evaluation_reports\figures\mesh_snapshots\moderate\0141.png


C:\Users\anshk\AppData\Local\Temp\ipykernel_20792\924178794.py:33: PyVistaFutureWarning: The default value of `algorithm` for the filter
`UnstructuredGrid.extract_surface` will change in the future. It currently defaults to
`'dataset_surface'`, but will change to `None`. Explicitly set the `algorithm` keyword to
silence this warning.
  surface = grid.extract_surface().triangulate()


[69/69] data\evaluation_reports\figures\mesh_snapshots\poor\0141.png
Saved 69 comparison images and 207 panel index rows.
Index: data\evaluation_reports\figures\mesh_snapshots\snapshot_index.csv


## Did the GNN help beyond rule-targeted correction?

The table below compares paired test rows (`GNN - rule_targeted`) within each phantom and tier. A win is defined as lower poor-element percentage, higher minimum scaled Jacobian, or fewer FE iterations. Counts are shown alongside median paired changes so improvements and regressions are not hidden by a single aggregate.

In [3]:
paired = selected[selected["method"].isin(["rule_targeted", "gnn"])].pivot(
    index=["id", "tier"], columns="method",
    values=["poor_pct", "min_sj", "fe_iters"],
)
if paired.isna().any().any():
    raise ValueError("Paired GNN/rule-targeted metrics contain missing values")

summary_rows = []
for tier in TIERS:
    tier_pair = paired.xs(tier, level="tier")
    d_poor = tier_pair[("poor_pct", "gnn")] - tier_pair[("poor_pct", "rule_targeted")]
    d_sj = tier_pair[("min_sj", "gnn")] - tier_pair[("min_sj", "rule_targeted")]
    d_fe = tier_pair[("fe_iters", "gnn")] - tier_pair[("fe_iters", "rule_targeted")]
    for metric, delta, improves in (
        ("poor_pct", d_poor, lambda values: values < 0),
        ("min_sj", d_sj, lambda values: values > 0),
        ("fe_iters", d_fe, lambda values: values < 0),
    ):
        improved = improves(delta)
        unchanged = np.isclose(delta, 0)
        summary_rows.append({
            "tier": tier,
            "metric": metric,
            "median_rule_targeted": float(tier_pair[(metric, "rule_targeted")].median()),
            "median_gnn": float(tier_pair[(metric, "gnn")].median()),
            "median_delta_gnn_minus_rule": float(delta.median()),
            "gnn_wins": int(improved.sum()),
            "ties": int(unchanged.sum()),
            "gnn_losses": int((~improved & ~unchanged).sum()),
            "n": int(len(delta)),
        })

comparison_summary = pd.DataFrame(summary_rows)
summary_path = SNAPSHOT_ROOT / "gnn_vs_rule_targeted_summary.csv"
comparison_summary.to_csv(summary_path, index=False)
display(comparison_summary.round(4))
print(f"Saved paired summary: {summary_path.relative_to(PROJECT_ROOT)}")

classical_summary = (
    selected[selected["method"].isin(METHODS)]
    .groupby(["tier", "method"])[["poor_pct", "min_sj", "fe_iters"]]
    .median()
    .round(4)
)
print("Per-tier medians for baseline and both corrections:")
display(classical_summary)

,tier,metric,median_rule_targeted,median_gnn,median_delta_gnn_minus_rule,gnn_wins,ties,gnn_losses,n
0,clean,poor_pct,16.0810,14.1850,-1.890,23,0,0,23
1,clean,min_sj,0.0183,0.0184,0.000,6,14,3,23
2,clean,fe_iters,251.0000,250.0000,-2.000,20,3,0,23
3,moderate,poor_pct,16.6190,15.0490,-1.501,23,0,0,23
4,moderate,min_sj,0.0179,0.0180,0.000,4,15,4,23
5,moderate,fe_iters,259.0000,258.0000,-1.000,17,4,2,23
6,poor,poor_pct,15.2610,14.6320,-0.564,23,0,0,23
7,poor,min_sj,0.0164,0.0164,0.000,0,19,4,23
8,poor,fe_iters,277.0000,276.0000,-1.000,12,7,4,23


Saved paired summary: data\evaluation_reports\figures\mesh_snapshots\gnn_vs_rule_targeted_summary.csv
Per-tier medians for baseline and both corrections:


poor_pct  min_sj  fe_iters
tier     method                                   
clean    classical        12.437  0.0135     254.0
         gnn              14.185  0.0184     250.0
         rule_targeted    16.081  0.0183     251.0
moderate classical        13.690  0.0137     261.0
         gnn              15.049  0.0180     258.0
         rule_targeted    16.619  0.0179     259.0
poor     classical        14.552  0.0121     280.0
         gnn              14.632  0.0164     276.0
         rule_targeted    15.261  0.0164     277.0